In [1]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
# =============================================================================
# 11_create_weekly_satellite_data.ipynb
#
# PURPOSE
# =============================================================================
#
# Create 7-day Sentinel-1 and Sentinel-2 composite imagery from the
# acquisition-level satellite dataset created by Notebook 10.
#
#
# SAMPLE
# =============================================================================
#
# Notebook 10 defines the sample:
#
#   - 10 treatment units
#   - 5 matched counterfactual units per treatment
#   - 50 counterfactual units
#   - 60 total spatial units
#
#
# This notebook DOES NOT independently select sites.
#
# It reads:
#
#   finals/daily_datasets/selected_site_sample.csv
#
# This guarantees that the daily and weekly datasets use exactly
# the same treatment-control sample.
#
#
# SOURCE DATA
# =============================================================================
#
# finals/
# └── daily_datasets/
#
#     ├── sentinel1/
#     ├── sentinel2/
#     ├── selected_site_sample.csv
#     └── daily_satellite_inventory.csv
#
#
# This notebook DOES NOT:
#
#   - connect to Earth Engine
#   - download satellite imagery
#   - modify acquisition-level TIFFs
#   - redefine treatment/control matching
#
#
# TEMPORAL DESIGN
# =============================================================================
#
# BEFORE:
#
#   2024-05-10 through 2024-09-26
#
#   140 calendar days
#   exactly 20 × 7-day periods
#
#
# AFTER:
#
#   2024-09-27 through 2025-02-13
#
#   140 calendar days
#   exactly 20 × 7-day periods
#
#
# September 27 is the first AFTER date.
#
#
# Fixed panel:
#
#   before_W01 ... before_W20
#
#   after_W01  ... after_W20
#
#
# COMPOSITING RULE
# =============================================================================
#
# For every:
#
#   site × sensor × 7-day period
#
#
# 0 acquisitions:
#
#   - no TIFF is created
#   - site-period remains in the quality table as missing
#
#
# 1 acquisition:
#
#   - weekly TIFF equals the single acquisition
#
#
# 2+ acquisitions:
#
#   - pixel-wise NaN-aware median
#
#
# PRIMARY QUALITY DEFINITION
# =============================================================================
#
# A spatial pixel is VALID when AT LEAST ONE output band has a finite value.
#
# Primary:
#
#   valid_pixel_fraction
#   valid_pixel_percentage
#
#
# Diagnostic:
#
#   valid_pixel_fraction_any_band
#   valid_pixel_fraction_all_bands
#
#
# RESUME-SAFE BEHAVIOR
# =============================================================================
#
# Existing valid weekly TIFF:
#
#   -> do not rebuild
#   -> recalculate quality locally
#   -> build_action = skipped_existing
#
#
# Existing invalid TIFF:
#
#   -> rename *.invalid
#   -> rebuild from daily TIFFs
#
#
# Missing TIFF:
#
#   -> create from daily TIFFs
#
#
# OUTPUT STRUCTURE
# =============================================================================
#
# finals/
# └── weekly_datasets/
#
#     ├── sentinel1/
#     │   ├── treatment/
#     │   │   ├── before/
#     │   │   └── after/
#     │   └── counterfactual/
#     │       ├── before/
#     │       └── after/
#     │
#     ├── sentinel2/
#     │   ├── treatment/
#     │   │   ├── before/
#     │   │   └── after/
#     │   └── counterfactual/
#     │       ├── before/
#     │       └── after/
#     │
#     ├── weekly_image_quality.csv
#     ├── weekly_image_quality.xlsx
#     ├── weekly_quality_summary.csv
#     ├── weekly_period_definitions.csv
#     ├── weekly_period_dimension.csv
#     ├── weekly_site_dimension.csv
#     ├── weekly_build_action_summary.csv
#     ├── weekly_folder_summary.csv
#     └── weekly_sample_validation.csv
#
# =============================================================================


# =============================================================================
# 1. Packages
# =============================================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import rasterio


warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
)


print(
    "Packages loaded successfully."
)


# =============================================================================
# 2. Project paths
# =============================================================================

BASE_DIR = Path(
    "/Users/gaoyujuan/REAP Dropbox/Gao yujuan/"
    "Virginia Tech/CALS/datasets"
)


FINALS_DIR = (
    BASE_DIR /
    "finals"
)


# -----------------------------------------------------------------------------
# Notebook 10 source
# -----------------------------------------------------------------------------

DAILY_DIR = (
    FINALS_DIR /
    "daily_datasets"
)


DAILY_INVENTORY_FILE = (
    DAILY_DIR /
    "daily_satellite_inventory.csv"
)


SELECTED_SAMPLE_FILE = (
    DAILY_DIR /
    "selected_site_sample.csv"
)


# -----------------------------------------------------------------------------
# Weekly output
# -----------------------------------------------------------------------------

WEEKLY_DIR = (
    FINALS_DIR /
    "weekly_datasets"
)


S1_WEEKLY_DIR = (
    WEEKLY_DIR /
    "sentinel1"
)


S2_WEEKLY_DIR = (
    WEEKLY_DIR /
    "sentinel2"
)


QUALITY_CSV_FILE = (
    WEEKLY_DIR /
    "weekly_image_quality.csv"
)


QUALITY_EXCEL_FILE = (
    WEEKLY_DIR /
    "weekly_image_quality.xlsx"
)


SUMMARY_CSV_FILE = (
    WEEKLY_DIR /
    "weekly_quality_summary.csv"
)


PERIOD_DEFINITION_FILE = (
    WEEKLY_DIR /
    "weekly_period_definitions.csv"
)


PERIOD_DIMENSION_FILE = (
    WEEKLY_DIR /
    "weekly_period_dimension.csv"
)


SITE_DIMENSION_FILE = (
    WEEKLY_DIR /
    "weekly_site_dimension.csv"
)


BUILD_ACTION_FILE = (
    WEEKLY_DIR /
    "weekly_build_action_summary.csv"
)


FOLDER_SUMMARY_FILE = (
    WEEKLY_DIR /
    "weekly_folder_summary.csv"
)


SAMPLE_VALIDATION_FILE = (
    WEEKLY_DIR /
    "weekly_sample_validation.csv"
)


WEEKLY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# 3. Check Notebook 10 outputs
# =============================================================================

required_files = [

    DAILY_INVENTORY_FILE,

    SELECTED_SAMPLE_FILE,

]


missing_files = [

    file_path

    for file_path in required_files

    if not file_path.exists()

]


if missing_files:

    raise FileNotFoundError(
        "Required Notebook 10 outputs are missing:\n\n"
        +
        "\n".join(
            str(file_path)
            for file_path in missing_files
        )
        +
        "\n\nRun Notebook 10 first."
    )


print(
    "\nNotebook 10 daily inventory:"
)


print(
    DAILY_INVENTORY_FILE
)


print(
    "\nNotebook 10 selected sample:"
)


print(
    SELECTED_SAMPLE_FILE
)


# =============================================================================
# 4. Study calendar
# =============================================================================

HELENE_REFERENCE_DATE = pd.Timestamp(
    "2024-09-27"
)


BEFORE_START = pd.Timestamp(
    "2024-05-10"
)


BEFORE_END = pd.Timestamp(
    "2024-09-26"
)


AFTER_START = pd.Timestamp(
    "2024-09-27"
)


AFTER_END = pd.Timestamp(
    "2025-02-13"
)


STUDY_START = BEFORE_START


STUDY_END = AFTER_END


print(
    "\nStudy period:"
)


print(
    STUDY_START.strftime("%Y-%m-%d"),
    "through",
    STUDY_END.strftime("%Y-%m-%d"),
)


print(
    "\nBefore:"
)


print(
    BEFORE_START.strftime("%Y-%m-%d"),
    "through",
    BEFORE_END.strftime("%Y-%m-%d"),
)


print(
    "\nAfter:"
)


print(
    AFTER_START.strftime("%Y-%m-%d"),
    "through",
    AFTER_END.strftime("%Y-%m-%d"),
)


# =============================================================================
# 5. Settings
# =============================================================================

SKIP_EXISTING_WEEKLY = True


# =============================================================================
# 6. Expected sample structure
# =============================================================================

EXPECTED_TREATMENT_SITES = 10


EXPECTED_CONTROLS_PER_TREATMENT = 5


EXPECTED_COUNTERFACTUAL_SITES = (
    EXPECTED_TREATMENT_SITES
    *
    EXPECTED_CONTROLS_PER_TREATMENT
)


EXPECTED_TOTAL_SITES = (
    EXPECTED_TREATMENT_SITES
    +
    EXPECTED_COUNTERFACTUAL_SITES
)


EXPECTED_WEEKS_BEFORE = 20


EXPECTED_WEEKS_AFTER = 20


EXPECTED_WEEKS_TOTAL = (
    EXPECTED_WEEKS_BEFORE
    +
    EXPECTED_WEEKS_AFTER
)


# =============================================================================
# 7. Sensor bands
# =============================================================================

SENSOR_BANDS = {

    "sentinel1": [

        "VV",

        "VH",

        "VV_minus_VH",

    ],

    "sentinel2": [

        "B2",

        "B3",

        "B4",

        "B8",

        "B11",

        "B12",

        "NDVI",

        "NDWI",

    ],

}


# =============================================================================
# 8. Create output folders
# =============================================================================

for sensor_root in [

    S1_WEEKLY_DIR,

    S2_WEEKLY_DIR,

]:

    for group in [

        "treatment",

        "counterfactual",

    ]:

        for period in [

            "before",

            "after",

        ]:

            folder = (
                sensor_root /
                group /
                period
            )


            folder.mkdir(
                parents=True,
                exist_ok=True,
            )


print(
    "\nWeekly output directory:"
)


print(
    WEEKLY_DIR
)


# =============================================================================
# 9. Load selected Notebook 10 sample
# =============================================================================

selected_sample = pd.read_csv(
    SELECTED_SAMPLE_FILE
)


required_sample_columns = [

    "site_id",

    "group",

    "matched_treatment_site_id",

    "control_rank",

]


missing_sample_columns = [

    column

    for column in required_sample_columns

    if column not in selected_sample.columns

]


if missing_sample_columns:

    raise ValueError(
        "selected_site_sample.csv is missing columns:\n"
        +
        str(
            missing_sample_columns
        )
    )


selected_sample[
    "site_id"
] = (
    selected_sample[
        "site_id"
    ]
    .astype(str)
)


selected_sample[
    "group"
] = (
    selected_sample[
        "group"
    ]
    .astype(str)
)


# Keep missing treatment match IDs as NaN rather than literal "nan".
selected_sample[
    "matched_treatment_site_id"
] = (
    selected_sample[
        "matched_treatment_site_id"
    ]
    .where(
        selected_sample[
            "matched_treatment_site_id"
        ]
        .notna(),
        np.nan,
    )
)


selected_sample[
    "matched_treatment_site_id"
] = (
    selected_sample[
        "matched_treatment_site_id"
    ]
    .astype("string")
)


selected_sample[
    "control_rank"
] = pd.to_numeric(
    selected_sample[
        "control_rank"
    ],
    errors=
        "coerce",
)


# =============================================================================
# 10. Ensure treatment rows map to themselves
# =============================================================================

treatment_mask = (
    selected_sample[
        "group"
    ]
    ==
    "treatment"
)


selected_sample.loc[
    treatment_mask,
    "matched_treatment_site_id",
] = (
    selected_sample.loc[
        treatment_mask,
        "site_id",
    ]
    .astype(str)
)


# =============================================================================
# 11. Validate selected sample
# =============================================================================

treatment_sample = (
    selected_sample
    .loc[
        selected_sample[
            "group"
        ]
        ==
        "treatment"
    ]
    .copy()
)


counterfactual_sample = (
    selected_sample
    .loc[
        selected_sample[
            "group"
        ]
        ==
        "counterfactual"
    ]
    .copy()
)


treatment_count = (
    treatment_sample[
        "site_id"
    ]
    .nunique()
)


counterfactual_count = (
    counterfactual_sample[
        "site_id"
    ]
    .nunique()
)


print(
    "\n"
    + "=" * 100
)


print(
    "SELECTED SAMPLE"
)


print(
    "=" * 100
)


print(
    "\nTreatment sites:",
    treatment_count
)


print(
    "Counterfactual sites:",
    counterfactual_count
)


print(
    "Total sites:",
    selected_sample[
        "site_id"
    ].nunique()
)


# =============================================================================
# 12. Validate 5 controls per treatment
# =============================================================================

controls_per_treatment = (
    counterfactual_sample
    .groupby(
        "matched_treatment_site_id"
    )[
        "site_id"
    ]
    .nunique()
    .rename(
        "counterfactual_count"
    )
    .reset_index()
)


sample_validation = pd.DataFrame(
    {

        "matched_treatment_site_id":
            treatment_sample[
                "site_id"
            ]
            .drop_duplicates()
            .astype(str)
            .tolist(),

    }
)


sample_validation = (
    sample_validation
    .merge(
        controls_per_treatment,
        on=
            "matched_treatment_site_id",
        how=
            "left",
    )
)


sample_validation[
    "counterfactual_count"
] = (
    sample_validation[
        "counterfactual_count"
    ]
    .fillna(0)
    .astype(int)
)


sample_validation[
    "expected_counterfactual_count"
] = (
    EXPECTED_CONTROLS_PER_TREATMENT
)


sample_validation[
    "sample_complete"
] = (
    sample_validation[
        "counterfactual_count"
    ]
    ==
    EXPECTED_CONTROLS_PER_TREATMENT
)


sample_validation.to_csv(
    SAMPLE_VALIDATION_FILE,
    index=False,
)


print(
    "\nControls per treatment:"
)


print(
    sample_validation.to_string(
        index=False
    )
)


# =============================================================================
# 13. Generate exactly 20 before + 20 after 7-day periods
# =============================================================================

def create_7day_periods(
    period_name,
    start_date,
    number_of_periods=20,
):

    records = []


    for week_number in range(
        1,
        number_of_periods + 1,
    ):

        period_start = (
            start_date
            +
            pd.Timedelta(
                days=
                    (
                        week_number - 1
                    )
                    *
                    7
            )
        )


        period_end = (
            period_start
            +
            pd.Timedelta(
                days=6
            )
        )


        records.append(
            {

                "period":
                    period_name,

                "week_number":
                    week_number,

                "week_id":
                    (
                        f"{period_name}_"
                        f"W{week_number:02d}"
                    ),

                "period_start":
                    period_start,

                "period_end":
                    period_end,

                "calendar_days":
                    7,

            }
        )


    return records


period_definitions = pd.DataFrame(

    create_7day_periods(
        "before",
        BEFORE_START,
        EXPECTED_WEEKS_BEFORE,
    )
    +
    create_7day_periods(
        "after",
        AFTER_START,
        EXPECTED_WEEKS_AFTER,
    )

)


# =============================================================================
# 14. Validate weekly period boundaries
# =============================================================================

before_definition = (
    period_definitions
    .loc[
        period_definitions[
            "period"
        ]
        ==
        "before"
    ]
    .copy()
)


after_definition = (
    period_definitions
    .loc[
        period_definitions[
            "period"
        ]
        ==
        "after"
    ]
    .copy()
)


assert len(
    before_definition
) == 20


assert len(
    after_definition
) == 20


assert (
    before_definition.iloc[
        0
    ][
        "period_start"
    ]
    ==
    BEFORE_START
)


assert (
    before_definition.iloc[
        -1
    ][
        "period_end"
    ]
    ==
    BEFORE_END
)


assert (
    after_definition.iloc[
        0
    ][
        "period_start"
    ]
    ==
    AFTER_START
)


assert (
    after_definition.iloc[
        -1
    ][
        "period_end"
    ]
    ==
    AFTER_END
)


period_definitions.to_csv(
    PERIOD_DEFINITION_FILE,
    index=False,
)


print(
    "\nWeekly periods:"
)


print(
    period_definitions[
        [
            "week_id",
            "period_start",
            "period_end",
        ]
    ]
    .to_string(
        index=False
    )
)


# =============================================================================
# 15. Load Notebook 10 daily inventory
# =============================================================================

inventory = pd.read_csv(
    DAILY_INVENTORY_FILE
)


required_columns = [

    "site_id",

    "group",

    "sensor",

    "acquisition_date",

    "file_path",

]


missing_columns = [

    column

    for column in required_columns

    if column not in inventory.columns

]


if missing_columns:

    raise ValueError(
        "Notebook 10 inventory is missing required columns:\n"
        +
        str(
            missing_columns
        )
    )


inventory[
    "site_id"
] = (
    inventory[
        "site_id"
    ]
    .astype(str)
)


inventory[
    "group"
] = (
    inventory[
        "group"
    ]
    .astype(str)
)


inventory[
    "sensor"
] = (
    inventory[
        "sensor"
    ]
    .astype(str)
)


inventory[
    "acquisition_date"
] = pd.to_datetime(
    inventory[
        "acquisition_date"
    ]
)


print(
    "\nRaw daily inventory rows:"
)


print(
    len(
        inventory
    )
)


# =============================================================================
# 16. Restrict inventory to selected sample
# =============================================================================

selected_site_ids = set(
    selected_sample[
        "site_id"
    ]
    .astype(str)
)


inventory = (
    inventory
    .loc[
        inventory[
            "site_id"
        ]
        .isin(
            selected_site_ids
        )
    ]
    .copy()
)


# =============================================================================
# 17. Attach matching metadata
# =============================================================================

sample_metadata = (
    selected_sample[
        [
            "site_id",
            "group",
            "matched_treatment_site_id",
            "control_rank",
        ]
    ]
    .drop_duplicates()
    .copy()
)


for column in [

    "matched_treatment_site_id",

    "control_rank",

]:

    if column in inventory.columns:

        inventory = (
            inventory
            .drop(
                columns=
                    column
            )
        )


inventory = (
    inventory
    .merge(
        sample_metadata,
        on=[
            "site_id",
            "group",
        ],
        how=
            "left",
    )
)


# =============================================================================
# 18. Keep successful / existing daily TIFFs
# =============================================================================

if "status" in inventory.columns:

    inventory = (
        inventory
        .loc[
            inventory[
                "status"
            ]
            .isin(
                [
                    "success",
                    "existing",
                ]
            )
        ]
        .copy()
    )


# =============================================================================
# 19. Restrict to study period
# =============================================================================

inventory = (
    inventory
    .loc[
        (
            inventory[
                "acquisition_date"
            ]
            >=
            STUDY_START
        )
        &
        (
            inventory[
                "acquisition_date"
            ]
            <=
            STUDY_END
        )
    ]
    .copy()
)


# =============================================================================
# 20. Check source TIFF files
# =============================================================================

inventory[
    "file_exists"
] = (
    inventory[
        "file_path"
    ]
    .astype(str)
    .apply(
        lambda path:
            Path(
                path
            ).exists()
    )
)


missing_source_files = (
    inventory
    .loc[
        ~inventory[
            "file_exists"
        ]
    ]
    .copy()
)


if not missing_source_files.empty:

    print(
        "\nWARNING:"
    )


    print(
        len(
            missing_source_files
        ),
        "daily inventory rows point to missing TIFFs."
    )


    print(
        "Those acquisitions will not be used."
    )


inventory = (
    inventory
    .loc[
        inventory[
            "file_exists"
        ]
    ]
    .copy()
)


print(
    "\nUsable daily acquisitions:"
)


print(
    len(
        inventory
    )
)


# =============================================================================
# 21. Read raster
# =============================================================================

def read_raster(
    file_path,
):

    file_path = Path(
        file_path
    )


    with rasterio.open(
        file_path
    ) as src:

        data = (
            src
            .read(
                masked=True
            )
            .astype(
                "float32"
            )
            .filled(
                np.nan
            )
        )


        return {

            "data":
                data,

            "profile":
                src.profile.copy(),

            "transform":
                src.transform,

            "crs":
                src.crs,

            "width":
                int(
                    src.width
                ),

            "height":
                int(
                    src.height
                ),

            "band_count":
                int(
                    src.count
                ),

        }


# =============================================================================
# 22. Inspect TIFF
# =============================================================================

def inspect_tiff(
    file_path,
    band_names,
):

    file_path = Path(
        file_path
    )


    if not file_path.exists():

        return {

            "reusable":
                False,

            "validation_status":
                "missing",

            "error":
                "File does not exist.",

        }


    try:

        raster = (
            read_raster(
                file_path
            )
        )


        if (
            raster[
                "band_count"
            ]
            !=
            len(
                band_names
            )
        ):

            return {

                "reusable":
                    False,

                "validation_status":
                    "wrong_band_count",

                "error":
                    (
                        f"Expected {len(band_names)} bands, "
                        f"found {raster['band_count']}."
                    ),

            }


        if (
            raster[
                "width"
            ]
            <=
            0
            or
            raster[
                "height"
            ]
            <=
            0
        ):

            return {

                "reusable":
                    False,

                "validation_status":
                    "invalid_dimensions",

                "error":
                    "Invalid raster dimensions.",

            }


        if raster[
            "crs"
        ] is None:

            return {

                "reusable":
                    False,

                "validation_status":
                    "missing_crs",

                "error":
                    "Raster CRS missing.",

            }


        data = (
            raster[
                "data"
            ]
        )


        finite = np.isfinite(
            data
        )


        # ---------------------------------------------------------------------
        # PRIMARY:
        # at least one output band is valid.
        # ---------------------------------------------------------------------

        valid_any = (
            finite
            .any(
                axis=0
            )
        )


        # ---------------------------------------------------------------------
        # Strict diagnostic:
        # all output bands are valid.
        # ---------------------------------------------------------------------

        valid_all = (
            finite
            .all(
                axis=0
            )
        )


        result = {

            "reusable":
                True,

            "validation_status":
                "valid",

            "error":
                None,

            "valid_pixel_fraction":
                float(
                    valid_any.mean()
                ),

            "valid_pixel_percentage":
                float(
                    valid_any.mean()
                    *
                    100
                ),

            "valid_pixel_fraction_any_band":
                float(
                    valid_any.mean()
                ),

            "valid_pixel_percentage_any_band":
                float(
                    valid_any.mean()
                    *
                    100
                ),

            "valid_pixel_fraction_all_bands":
                float(
                    valid_all.mean()
                ),

            "valid_pixel_percentage_all_bands":
                float(
                    valid_all.mean()
                    *
                    100
                ),

        }


        for band_index, band_name in enumerate(
            band_names
        ):

            fraction = float(
                finite[
                    band_index
                ]
                .mean()
            )


            result[
                f"valid_fraction_{band_name}"
            ] = fraction


            result[
                f"valid_percentage_{band_name}"
            ] = (
                fraction
                *
                100
            )


        return result


    except Exception as error:

        return {

            "reusable":
                False,

            "validation_status":
                "corrupt_or_unreadable",

            "error":
                str(
                    error
                ),

        }


# =============================================================================
# 23. Raster compatibility
# =============================================================================

def check_raster_compatibility(
    raster_infos,
):

    if len(
        raster_infos
    ) <= 1:

        return (
            True,
            None,
        )


    reference = (
        raster_infos[
            0
        ]
    )


    for image_number, current in enumerate(
        raster_infos[
            1:
        ],
        start=2,
    ):

        if (
            current[
                "data"
            ].shape
            !=
            reference[
                "data"
            ].shape
        ):

            return (

                False,

                (
                    f"Image {image_number} shape mismatch: "
                    f"{current['data'].shape} vs "
                    f"{reference['data'].shape}"
                ),

            )


        if (
            str(
                current[
                    "crs"
                ]
            )
            !=
            str(
                reference[
                    "crs"
                ]
            )
        ):

            return (

                False,

                f"Image {image_number} CRS mismatch.",

            )


        if (
            current[
                "transform"
            ]
            !=
            reference[
                "transform"
            ]
        ):

            return (

                False,

                f"Image {image_number} pixel-grid mismatch.",

            )


    return (
        True,
        None,
    )


# =============================================================================
# 24. Create weekly composite
# =============================================================================

def create_weekly_composite(
    source_files,
):

    raster_infos = [

        read_raster(
            file_path
        )

        for file_path in source_files

    ]


    compatible, error = (
        check_raster_compatibility(
            raster_infos
        )
    )


    if not compatible:

        raise ValueError(
            error
        )


    # -------------------------------------------------------------------------
    # One acquisition
    # -------------------------------------------------------------------------

    if len(
        raster_infos
    ) == 1:

        return (

            raster_infos[
                0
            ][
                "data"
            ].copy(),

            raster_infos[
                0
            ],

        )


    # -------------------------------------------------------------------------
    # Multiple acquisitions
    # -------------------------------------------------------------------------

    stack = np.stack(

        [

            raster[
                "data"
            ]

            for raster in raster_infos

        ],

        axis=0,

    )


    with warnings.catch_warnings():

        warnings.simplefilter(
            "ignore",
            category=RuntimeWarning,
        )


        composite = np.nanmedian(
            stack,
            axis=0,
        )


    return (
        composite,
        raster_infos[
            0
        ],
    )


# =============================================================================
# 25. Save weekly composite
# =============================================================================

def save_weekly_composite(
    composite,
    reference_info,
    output_file,
):

    output_file = Path(
        output_file
    )


    output_file.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    profile = (
        reference_info[
            "profile"
        ]
        .copy()
    )


    profile.update(
        {

            "driver":
                "GTiff",

            "dtype":
                "float32",

            "count":
                int(
                    composite.shape[
                        0
                    ]
                ),

            "compress":
                "deflate",

            "nodata":
                np.nan,

        }
    )


    with rasterio.open(
        output_file,
        "w",
        **profile,
    ) as dst:

        dst.write(
            composite.astype(
                "float32"
            )
        )


# =============================================================================
# 26. Invalid backup path
# =============================================================================

def get_invalid_backup_path(
    file_path,
):

    file_path = Path(
        file_path
    )


    candidate = (
        file_path
        .with_name(
            file_path.name
            +
            ".invalid"
        )
    )


    counter = 1


    while candidate.exists():

        candidate = (
            file_path
            .with_name(
                file_path.name
                +
                f".invalid_{counter}"
            )
        )


        counter += 1


    return candidate


# =============================================================================
# 27. Quality label
# =============================================================================

def quality_label(
    valid_fraction,
):

    if pd.isna(
        valid_fraction
    ):

        return "missing"


    if valid_fraction >= 0.80:

        return "excellent"


    if valid_fraction >= 0.50:

        return "usable"


    if valid_fraction >= 0.20:

        return "limited"


    if valid_fraction > 0:

        return "poor"


    return "unusable"


# =============================================================================
# 28. Source quality variable
# =============================================================================

def identify_source_quality_column(
    dataframe,
):

    for column in [

        "valid_pixel_fraction",

        "valid_pixel_fraction_any_band",

        "valid_pixel_fraction_all_bands",

    ]:

        if column in dataframe.columns:

            return column


    return None


SOURCE_QUALITY_COLUMN = (
    identify_source_quality_column(
        inventory
    )
)


print(
    "\nSource quality variable:"
)


print(
    SOURCE_QUALITY_COLUMN
)


# =============================================================================
# 29. Build / reuse weekly image
# =============================================================================

def build_or_reuse_weekly(
    source_files,
    output_file,
    band_names,
):

    output_file = Path(
        output_file
    )


    # =========================================================================
    # Existing weekly TIFF
    # =========================================================================

    if (
        SKIP_EXISTING_WEEKLY
        and
        output_file.exists()
    ):

        inspection = (
            inspect_tiff(
                output_file,
                band_names,
            )
        )


        if inspection.get(
            "reusable",
            False
        ):

            print(
                "    Existing weekly TIFF valid — skipping rebuild."
            )


            inspection[
                "build_action"
            ] = (
                "skipped_existing"
            )


            inspection[
                "composite_created"
            ] = 1


            return inspection


        print(
            "    Existing weekly TIFF invalid:"
        )


        print(
            "   ",
            inspection.get(
                "validation_status"
            ),
            "|",
            inspection.get(
                "error"
            ),
        )


        invalid_backup = (
            get_invalid_backup_path(
                output_file
            )
        )


        try:

            output_file.rename(
                invalid_backup
            )


            print(
                "    Invalid weekly TIFF moved to:"
            )


            print(
                "   ",
                invalid_backup
            )


        except Exception:

            try:

                output_file.unlink()

            except Exception:

                pass


    # =========================================================================
    # Create new weekly composite
    # =========================================================================

    try:

        composite, reference_info = (
            create_weekly_composite(
                source_files
            )
        )


        save_weekly_composite(

            composite=
                composite,

            reference_info=
                reference_info,

            output_file=
                output_file,

        )


        inspection = (
            inspect_tiff(
                output_file,
                band_names,
            )
        )


        if not inspection.get(
            "reusable",
            False
        ):

            return {

                "composite_created":
                    0,

                "build_action":
                    "failed",

                "valid_pixel_fraction":
                    np.nan,

                "valid_pixel_percentage":
                    np.nan,

                "error":
                    inspection.get(
                        "error"
                    ),

            }


        inspection[
            "composite_created"
        ] = 1


        inspection[
            "build_action"
        ] = (
            "created"
        )


        return inspection


    except Exception as error:

        return {

            "composite_created":
                0,

            "build_action":
                "failed",

            "valid_pixel_fraction":
                np.nan,

            "valid_pixel_percentage":
                np.nan,

            "error":
                str(
                    error
                ),

        }


# =============================================================================
# 30. Build complete site × sensor master
#
# IMPORTANT:
#
# selected_site_sample.csv defines the panel.
#
# We do NOT derive the selected sample from observed acquisitions.
#
# Therefore:
#
# a selected site remains in the weekly panel even if it has zero
# acquisitions during one or more weeks.
# =============================================================================

sensor_master = pd.DataFrame(
    {

        "sensor": [

            "sentinel1",

            "sentinel2",

        ]

    }
)


selected_sample[
    "_key"
] = 1


sensor_master[
    "_key"
] = 1


site_sensor_master = (
    selected_sample
    .merge(
        sensor_master,
        on=
            "_key",
    )
    .drop(
        columns=
            "_key"
    )
)


print(
    "\nSite × sensor combinations:"
)


print(
    len(
        site_sensor_master
    )
)


print(
    "\nExpected:"
)


print(
    EXPECTED_TOTAL_SITES
    *
    2
)


# =============================================================================
# 31. Generate weekly composites
# =============================================================================

weekly_records = []


total_combinations = len(
    site_sensor_master
)


for combination_number, (_, combination) in enumerate(
    site_sensor_master.iterrows(),
    start=1,
):

    site_id = str(
        combination[
            "site_id"
        ]
    )


    group = str(
        combination[
            "group"
        ]
    )


    sensor = str(
        combination[
            "sensor"
        ]
    )


    matched_treatment_site_id = (
        combination[
            "matched_treatment_site_id"
        ]
    )


    control_rank = (
        combination[
            "control_rank"
        ]
    )


    print(
        "\n"
        + "=" * 100
    )


    print(
        f"{combination_number}/{total_combinations}"
    )


    print(
        site_id,
        "|",
        group,
        "|",
        sensor,
    )


    if group == "counterfactual":

        print(
            "Matched treatment:",
            matched_treatment_site_id,
            "| control rank:",
            control_rank,
        )


    print(
        "=" * 100
    )


    site_inventory = (
        inventory
        .loc[
            (
                inventory[
                    "site_id"
                ]
                ==
                site_id
            )
            &
            (
                inventory[
                    "group"
                ]
                ==
                group
            )
            &
            (
                inventory[
                    "sensor"
                ]
                ==
                sensor
            )
        ]
        .copy()
    )


    if sensor == "sentinel1":

        sensor_root = (
            S1_WEEKLY_DIR
        )


    elif sensor == "sentinel2":

        sensor_root = (
            S2_WEEKLY_DIR
        )


    else:

        continue


    band_names = (
        SENSOR_BANDS[
            sensor
        ]
    )


    # =========================================================================
    # 40 weekly periods for every site × sensor
    # =========================================================================

    for _, period_row in (
        period_definitions.iterrows()
    ):

        period = str(
            period_row[
                "period"
            ]
        )


        week_number = int(
            period_row[
                "week_number"
            ]
        )


        week_id = str(
            period_row[
                "week_id"
            ]
        )


        period_start = pd.Timestamp(
            period_row[
                "period_start"
            ]
        )


        period_end = pd.Timestamp(
            period_row[
                "period_end"
            ]
        )


        acquisitions = (
            site_inventory
            .loc[
                (
                    site_inventory[
                        "acquisition_date"
                    ]
                    >=
                    period_start
                )
                &
                (
                    site_inventory[
                        "acquisition_date"
                    ]
                    <=
                    period_end
                )
            ]
            .sort_values(
                "acquisition_date"
            )
            .copy()
        )


        acquisition_count = (
            len(
                acquisitions
            )
        )


        acquisition_dates = (
            acquisitions[
                "acquisition_date"
            ]
            .dt.strftime(
                "%Y-%m-%d"
            )
            .tolist()
        )


        source_files = (
            acquisitions[
                "file_path"
            ]
            .astype(str)
            .tolist()
        )


        # ---------------------------------------------------------------------
        # Source image quality
        # ---------------------------------------------------------------------

        source_mean_valid_fraction = np.nan


        source_best_valid_fraction = np.nan


        if (
            acquisition_count > 0
            and
            SOURCE_QUALITY_COLUMN is not None
        ):

            quality_values = (
                acquisitions[
                    SOURCE_QUALITY_COLUMN
                ]
                .dropna()
            )


            if len(
                quality_values
            ) > 0:

                source_mean_valid_fraction = float(
                    quality_values.mean()
                )


                source_best_valid_fraction = float(
                    quality_values.max()
                )


        # ---------------------------------------------------------------------
        # Base record
        # ---------------------------------------------------------------------

        record = {

            "site_id":
                site_id,

            "group":
                group,

            "matched_treatment_site_id":
                matched_treatment_site_id,

            "control_rank":
                control_rank,

            "sensor":
                sensor,

            "period":
                period,

            "week_number":
                week_number,

            "week_id":
                week_id,

            "period_start":
                period_start.strftime(
                    "%Y-%m-%d"
                ),

            "period_end":
                period_end.strftime(
                    "%Y-%m-%d"
                ),

            # Compatibility with Notebook 13
            "window_start":
                period_start.strftime(
                    "%Y-%m-%d"
                ),

            "window_end":
                period_end.strftime(
                    "%Y-%m-%d"
                ),

            "calendar_days":
                7,

            "acquisition_count":
                acquisition_count,

            "acquisition_dates":
                json.dumps(
                    acquisition_dates
                ),

            "source_files":
                json.dumps(
                    source_files
                ),

            "has_any_acquisition":
                int(
                    acquisition_count > 0
                ),

            "has_data":
                int(
                    acquisition_count > 0
                ),

            "has_multiple_acquisitions":
                int(
                    acquisition_count > 1
                ),

            "multiple_acquisitions":
                int(
                    acquisition_count > 1
                ),

            "source_mean_valid_pixel_fraction":
                source_mean_valid_fraction,

            "source_best_valid_pixel_fraction":
                source_best_valid_fraction,

            "composite_created":
                0,

            "build_action":
                "missing",

            "output_file":
                None,

            "valid_pixel_fraction":
                np.nan,

            "valid_pixel_percentage":
                np.nan,

            "valid_pixel_fraction_any_band":
                np.nan,

            "valid_pixel_percentage_any_band":
                np.nan,

            "valid_pixel_fraction_all_bands":
                np.nan,

            "valid_pixel_percentage_all_bands":
                np.nan,

            "quality_label":
                "missing",

            "improvement_vs_mean_source":
                np.nan,

            "improvement_vs_best_source":
                np.nan,

            "error":
                None,

        }


        print(
            "\n",
            week_id,
            " | ",
            period_start.strftime(
                "%Y-%m-%d"
            ),
            " to ",
            period_end.strftime(
                "%Y-%m-%d"
            ),
            " | acquisitions = ",
            acquisition_count,
            sep=""
        )


        # ---------------------------------------------------------------------
        # No observation
        # ---------------------------------------------------------------------

        if acquisition_count == 0:

            print(
                "    No acquisition in this weekly period."
            )


            weekly_records.append(
                record
            )


            continue


        # ---------------------------------------------------------------------
        # Output path
        # ---------------------------------------------------------------------

        output_file = (

            sensor_root /
            group /
            period /
            (
                f"{site_id}_"
                f"{week_id}_"
                f"{period_start.strftime('%Y-%m-%d')}_"
                f"{period_end.strftime('%Y-%m-%d')}_"
                f"weekly_"
                f"{sensor}.tif"
            )

        )


        # ---------------------------------------------------------------------
        # Build or reuse
        # ---------------------------------------------------------------------

        result = (
            build_or_reuse_weekly(

                source_files=
                    source_files,

                output_file=
                    output_file,

                band_names=
                    band_names,

            )
        )


        record.update(
            result
        )


        if (
            result.get(
                "composite_created",
                0
            )
            ==
            1
        ):

            record[
                "output_file"
            ] = str(
                output_file
            )


        # ---------------------------------------------------------------------
        # Quality
        # ---------------------------------------------------------------------

        if pd.notna(
            record.get(
                "valid_pixel_fraction"
            )
        ):

            current_quality = (
                record[
                    "valid_pixel_fraction"
                ]
            )


            record[
                "quality_label"
            ] = (
                quality_label(
                    current_quality
                )
            )


            if pd.notna(
                source_mean_valid_fraction
            ):

                record[
                    "improvement_vs_mean_source"
                ] = (
                    current_quality
                    -
                    source_mean_valid_fraction
                )


            if pd.notna(
                source_best_valid_fraction
            ):

                record[
                    "improvement_vs_best_source"
                ] = (
                    current_quality
                    -
                    source_best_valid_fraction
                )


            print(
                "    Valid pixels:",
                round(
                    record[
                        "valid_pixel_percentage"
                    ],
                    2,
                ),
                "%"
            )


            print(
                "    Quality:",
                record[
                    "quality_label"
                ]
            )


            print(
                "    Action:",
                record[
                    "build_action"
                ]
            )


        weekly_records.append(
            record
        )


# =============================================================================
# 32. Detailed weekly quality dataset
# =============================================================================

weekly_quality = pd.DataFrame(
    weekly_records
)


# =============================================================================
# 33. Quality thresholds
# =============================================================================

for threshold in [

    0.40,

    0.50,

    0.60,

    0.80,

    0.90,

]:

    threshold_pct = int(
        threshold
        *
        100
    )


    weekly_quality[
        f"quality_ge_{threshold_pct}pct"
    ] = (
        weekly_quality[
            "valid_pixel_fraction"
        ]
        >=
        threshold
    ).astype(int)


weekly_quality[
    "quality_100pct"
] = (
    weekly_quality[
        "valid_pixel_fraction"
    ]
    >=
    0.999999
).astype(int)


weekly_quality.to_csv(
    QUALITY_CSV_FILE,
    index=False,
)


print(
    "\nWeekly image-quality CSV:"
)


print(
    QUALITY_CSV_FILE
)


# =============================================================================
# 34. Build / reuse summary
# =============================================================================

build_action_summary = (
    weekly_quality[
        "build_action"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "build_action"
    )
    .reset_index(
        name=
            "image_count"
    )
)


build_action_summary.to_csv(
    BUILD_ACTION_FILE,
    index=False,
)


print(
    "\nWeekly build/reuse:"
)


print(
    build_action_summary.to_string(
        index=False
    )
)


# =============================================================================
# 35. Overall weekly quality summary
# =============================================================================

weekly_summary = (
    weekly_quality
    .groupby(
        [
            "sensor",
            "group",
            "period",
        ],
        as_index=False,
    )
    .agg(

        number_of_sites=(
            "site_id",
            "nunique",
        ),

        expected_site_periods=(
            "week_id",
            "count",
        ),

        periods_with_data=(
            "has_any_acquisition",
            "sum",
        ),

        periods_with_multiple_acquisitions=(
            "has_multiple_acquisitions",
            "sum",
        ),

        total_acquisitions=(
            "acquisition_count",
            "sum",
        ),

        mean_acquisitions_per_period=(
            "acquisition_count",
            "mean",
        ),

        mean_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "mean",
        ),

        median_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "median",
        ),

        min_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "min",
        ),

        max_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "max",
        ),

        periods_ge_40pct_valid=(
            "quality_ge_40pct",
            "sum",
        ),

        periods_ge_50pct_valid=(
            "quality_ge_50pct",
            "sum",
        ),

        periods_ge_60pct_valid=(
            "quality_ge_60pct",
            "sum",
        ),

        periods_ge_80pct_valid=(
            "quality_ge_80pct",
            "sum",
        ),

        periods_ge_90pct_valid=(
            "quality_ge_90pct",
            "sum",
        ),

        mean_improvement_vs_mean_source=(
            "improvement_vs_mean_source",
            "mean",
        ),

        mean_improvement_vs_best_source=(
            "improvement_vs_best_source",
            "mean",
        ),

    )
)


weekly_summary[
    "periods_without_data"
] = (
    weekly_summary[
        "expected_site_periods"
    ]
    -
    weekly_summary[
        "periods_with_data"
    ]
)


weekly_summary[
    "percent_periods_with_data"
] = (
    weekly_summary[
        "periods_with_data"
    ]
    /
    weekly_summary[
        "expected_site_periods"
    ]
    *
    100
)


# =============================================================================
# 36. Threshold percentages
# =============================================================================

for threshold in [

    40,

    50,

    60,

    80,

    90,

]:

    weekly_summary[
        f"percent_available_periods_ge_{threshold}pct"
    ] = np.where(

        weekly_summary[
            "periods_with_data"
        ]
        >
        0,

        weekly_summary[
            f"periods_ge_{threshold}pct_valid"
        ]
        /
        weekly_summary[
            "periods_with_data"
        ]
        *
        100,

        np.nan,

    )


# =============================================================================
# 37. Convert fractions to percentages
# =============================================================================

for source_column, target_column in [

    (
        "mean_valid_pixel_fraction",
        "mean_valid_pixel_percentage",
    ),

    (
        "median_valid_pixel_fraction",
        "median_valid_pixel_percentage",
    ),

    (
        "min_valid_pixel_fraction",
        "min_valid_pixel_percentage",
    ),

    (
        "max_valid_pixel_fraction",
        "max_valid_pixel_percentage",
    ),

]:

    weekly_summary[
        target_column
    ] = (
        weekly_summary[
            source_column
        ]
        *
        100
    )


weekly_summary[
    "mean_improvement_vs_mean_source_percentage_points"
] = (
    weekly_summary[
        "mean_improvement_vs_mean_source"
    ]
    *
    100
)


weekly_summary[
    "mean_improvement_vs_best_source_percentage_points"
] = (
    weekly_summary[
        "mean_improvement_vs_best_source"
    ]
    *
    100
)


weekly_summary.to_csv(
    SUMMARY_CSV_FILE,
    index=False,
)


# =============================================================================
# 38. PERIOD DIMENSION
#
# Within each weekly period, compare quality across sites.
# =============================================================================

period_dimension = (
    weekly_quality
    .groupby(
        [
            "sensor",
            "group",
            "period",
            "week_number",
            "week_id",
            "period_start",
            "period_end",
        ],
        as_index=False,
    )
    .agg(

        number_of_sites=(
            "site_id",
            "nunique",
        ),

        images_with_data=(
            "has_any_acquisition",
            "sum",
        ),

        mean_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "mean",
        ),

        median_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "median",
        ),

        min_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "min",
        ),

        max_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "max",
        ),

        images_ge_40pct=(
            "quality_ge_40pct",
            "sum",
        ),

        images_ge_50pct=(
            "quality_ge_50pct",
            "sum",
        ),

        images_ge_60pct=(
            "quality_ge_60pct",
            "sum",
        ),

        images_ge_80pct=(
            "quality_ge_80pct",
            "sum",
        ),

        images_ge_90pct=(
            "quality_ge_90pct",
            "sum",
        ),

    )
)


for threshold in [

    40,

    50,

    60,

    80,

    90,

]:

    period_dimension[
        f"percent_images_ge_{threshold}pct"
    ] = np.where(

        period_dimension[
            "images_with_data"
        ]
        >
        0,

        period_dimension[
            f"images_ge_{threshold}pct"
        ]
        /
        period_dimension[
            "images_with_data"
        ]
        *
        100,

        np.nan,

    )


for source_column, target_column in [

    (
        "mean_valid_pixel_fraction",
        "mean_valid_pixel_percentage",
    ),

    (
        "median_valid_pixel_fraction",
        "median_valid_pixel_percentage",
    ),

    (
        "min_valid_pixel_fraction",
        "min_valid_pixel_percentage",
    ),

    (
        "max_valid_pixel_fraction",
        "max_valid_pixel_percentage",
    ),

]:

    period_dimension[
        target_column
    ] = (
        period_dimension[
            source_column
        ]
        *
        100
    )


period_dimension.to_csv(
    PERIOD_DIMENSION_FILE,
    index=False,
)


# =============================================================================
# 39. SITE DIMENSION
#
# For each site, compare quality across its 20 BEFORE / 20 AFTER weeks.
# =============================================================================

site_dimension = (
    weekly_quality
    .groupby(
        [
            "sensor",
            "group",
            "site_id",
            "matched_treatment_site_id",
            "control_rank",
            "period",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(

        expected_periods=(
            "week_id",
            "count",
        ),

        periods_with_data=(
            "has_any_acquisition",
            "sum",
        ),

        mean_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "mean",
        ),

        median_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "median",
        ),

        min_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "min",
        ),

        max_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "max",
        ),

        images_ge_40pct=(
            "quality_ge_40pct",
            "sum",
        ),

        images_ge_50pct=(
            "quality_ge_50pct",
            "sum",
        ),

        images_ge_60pct=(
            "quality_ge_60pct",
            "sum",
        ),

        images_ge_80pct=(
            "quality_ge_80pct",
            "sum",
        ),

        images_ge_90pct=(
            "quality_ge_90pct",
            "sum",
        ),

    )
)


site_dimension[
    "periods_without_data"
] = (
    site_dimension[
        "expected_periods"
    ]
    -
    site_dimension[
        "periods_with_data"
    ]
)


for threshold in [

    40,

    50,

    60,

    80,

    90,

]:

    site_dimension[
        f"percent_images_ge_{threshold}pct"
    ] = np.where(

        site_dimension[
            "periods_with_data"
        ]
        >
        0,

        site_dimension[
            f"images_ge_{threshold}pct"
        ]
        /
        site_dimension[
            "periods_with_data"
        ]
        *
        100,

        np.nan,

    )


for source_column, target_column in [

    (
        "mean_valid_pixel_fraction",
        "mean_valid_pixel_percentage",
    ),

    (
        "median_valid_pixel_fraction",
        "median_valid_pixel_percentage",
    ),

    (
        "min_valid_pixel_fraction",
        "min_valid_pixel_percentage",
    ),

    (
        "max_valid_pixel_fraction",
        "max_valid_pixel_percentage",
    ),

]:

    site_dimension[
        target_column
    ] = (
        site_dimension[
            source_column
        ]
        *
        100
    )


site_dimension.to_csv(
    SITE_DIMENSION_FILE,
    index=False,
)


# =============================================================================
# 40. Quality definitions
# =============================================================================

quality_definitions = pd.DataFrame(
    {

        "variable": [

            "site_id",

            "matched_treatment_site_id",

            "control_rank",

            "acquisition_count",

            "valid_pixel_fraction",

            "valid_pixel_fraction_any_band",

            "valid_pixel_fraction_all_bands",

            "improvement_vs_mean_source",

            "improvement_vs_best_source",

            "build_action",

        ],

        "meaning": [

            (
                "Treatment or counterfactual site ID."
            ),

            (
                "Treatment site associated with this spatial unit. "
                "Treatment rows map to themselves."
            ),

            (
                "Counterfactual rank within the treatment match set."
            ),

            (
                "Number of actual acquisition-level TIFFs contributing "
                "to the 7-day weekly composite."
            ),

            (
                "PRIMARY metric: fraction of spatial pixels where at least "
                "one output band contains a finite value."
            ),

            (
                "Explicit any-band version of the primary quality metric."
            ),

            (
                "Strict diagnostic requiring every output band to be valid."
            ),

            (
                "Weekly valid-pixel fraction minus average quality of "
                "source acquisition images."
            ),

            (
                "Weekly valid-pixel fraction minus quality of the best "
                "source acquisition."
            ),

            (
                "created = new weekly TIFF; skipped_existing = valid TIFF "
                "already existed; missing = no acquisition; failed = error."
            ),

        ],

    }
)


# =============================================================================
# 41. Excel workbook
# =============================================================================

try:

    with pd.ExcelWriter(
        QUALITY_EXCEL_FILE,
        engine=
            "openpyxl",
    ) as writer:

        weekly_quality.to_excel(
            writer,
            sheet_name=
                "image_quality",
            index=False,
        )


        weekly_summary.to_excel(
            writer,
            sheet_name=
                "summary",
            index=False,
        )


        period_dimension.to_excel(
            writer,
            sheet_name=
                "period_dimension",
            index=False,
        )


        site_dimension.to_excel(
            writer,
            sheet_name=
                "site_dimension",
            index=False,
        )


        period_definitions.to_excel(
            writer,
            sheet_name=
                "period_definitions",
            index=False,
        )


        selected_sample.to_excel(
            writer,
            sheet_name=
                "selected_sample",
            index=False,
        )


        sample_validation.to_excel(
            writer,
            sheet_name=
                "sample_validation",
            index=False,
        )


        quality_definitions.to_excel(
            writer,
            sheet_name=
                "quality_definitions",
            index=False,
        )


        build_action_summary.to_excel(
            writer,
            sheet_name=
                "build_actions",
            index=False,
        )


    print(
        "\nExcel workbook saved:"
    )


    print(
        QUALITY_EXCEL_FILE
    )


except ModuleNotFoundError:

    print(
        "\nopenpyxl is not installed."
    )


    print(
        "CSV outputs were still generated."
    )


    print(
        "%pip install openpyxl"
    )


# =============================================================================
# 42. Folder summary
# =============================================================================

folder_records = []


for sensor_name, sensor_root in [

    (
        "sentinel1",
        S1_WEEKLY_DIR,
    ),

    (
        "sentinel2",
        S2_WEEKLY_DIR,
    ),

]:

    for group in [

        "treatment",

        "counterfactual",

    ]:

        for period in [

            "before",

            "after",

        ]:

            folder = (
                sensor_root /
                group /
                period
            )


            folder_records.append(
                {

                    "sensor":
                        sensor_name,

                    "group":
                        group,

                    "period":
                        period,

                    "folder":
                        str(
                            folder
                        ),

                    "weekly_tiff_count":
                        len(
                            list(
                                folder.glob(
                                    "*.tif"
                                )
                            )
                        ),

                }
            )


folder_summary = pd.DataFrame(
    folder_records
)


folder_summary.to_csv(
    FOLDER_SUMMARY_FILE,
    index=False,
)


# =============================================================================
# 43. Panel coverage validation
# =============================================================================

coverage_check = (
    weekly_quality
    .groupby(
        [
            "sensor",
            "group",
        ],
        as_index=False,
    )
    .agg(

        unique_sites=(
            "site_id",
            "nunique",
        ),

        total_site_period_rows=(
            "week_id",
            "count",
        ),

        periods_with_data=(
            "has_any_acquisition",
            "sum",
        ),

        composites_created=(
            "composite_created",
            "sum",
        ),

    )
)


print(
    "\n"
    + "=" * 100
)


print(
    "WEEKLY PANEL VALIDATION"
)


print(
    "=" * 100
)


print(
    coverage_check.to_string(
        index=False
    )
)


# =============================================================================
# 44. Expected panel dimensions
# =============================================================================

print(
    "\nExpected spatial sample:"
)


print(
    "Treatment:",
    EXPECTED_TREATMENT_SITES
)


print(
    "Counterfactual:",
    EXPECTED_COUNTERFACTUAL_SITES
)


print(
    "Total sites:",
    EXPECTED_TOTAL_SITES
)


print(
    "\nExpected periods per site:"
)


print(
    "20 before + 20 after = 40"
)


print(
    "\nExpected rows per sensor:"
)


print(
    EXPECTED_TOTAL_SITES
    *
    EXPECTED_WEEKS_TOTAL
)


print(
    "\nExpected rows for both sensors:"
)


print(
    EXPECTED_TOTAL_SITES
    *
    EXPECTED_WEEKS_TOTAL
    *
    2
)


# =============================================================================
# 45. Main quality summary
# =============================================================================

print(
    "\n"
    + "=" * 100
)


print(
    "WEEKLY IMAGE QUALITY SUMMARY"
)


print(
    "=" * 100
)


columns_to_show = [

    "sensor",

    "group",

    "period",

    "number_of_sites",

    "expected_site_periods",

    "periods_with_data",

    "periods_with_multiple_acquisitions",

    "mean_acquisitions_per_period",

    "mean_valid_pixel_percentage",

    "median_valid_pixel_percentage",

    "periods_ge_80pct_valid",

    "percent_available_periods_ge_80pct",

]


print(
    weekly_summary[
        columns_to_show
    ]
    .to_string(
        index=False
    )
)


# =============================================================================
# 46. Sentinel-2 weekly period quality
# =============================================================================

s2_period = (
    period_dimension
    .loc[
        period_dimension[
            "sensor"
        ]
        ==
        "sentinel2"
    ]
    .copy()
)


print(
    "\n"
    + "=" * 100
)


print(
    "SENTINEL-2 WEEKLY PERIOD QUALITY"
)


print(
    "=" * 100
)


print(
    s2_period[
        [
            "group",
            "period",
            "week_id",
            "number_of_sites",
            "images_with_data",
            "mean_valid_pixel_percentage",
            "median_valid_pixel_percentage",
            "images_ge_80pct",
            "percent_images_ge_80pct",
        ]
    ]
    .to_string(
        index=False
    )
)


# =============================================================================
# 47. Temporal validation
# =============================================================================

print(
    "\n"
    + "=" * 100
)


print(
    "TEMPORAL VALIDATION"
)


print(
    "=" * 100
)


print(
    "\nBefore periods:",
    len(
        before_definition
    )
)


print(
    "After periods:",
    len(
        after_definition
    )
)


print(
    "\nFirst before:"
)


print(
    before_definition.iloc[
        0
    ][
        [
            "week_id",
            "period_start",
            "period_end",
        ]
    ]
)


print(
    "\nLast before:"
)


print(
    before_definition.iloc[
        -1
    ][
        [
            "week_id",
            "period_start",
            "period_end",
        ]
    ]
)


print(
    "\nFirst after:"
)


print(
    after_definition.iloc[
        0
    ][
        [
            "week_id",
            "period_start",
            "period_end",
        ]
    ]
)


print(
    "\nLast after:"
)


print(
    after_definition.iloc[
        -1
    ][
        [
            "week_id",
            "period_start",
            "period_end",
        ]
    ]
)


# =============================================================================
# 48. Final summary
# =============================================================================

print(
    "\n"
    + "=" * 100
)


print(
    "WEEKLY DATASET COMPLETE"
)


print(
    "=" * 100
)


print(
    "\nSource daily dataset:"
)


print(
    DAILY_DIR
)


print(
    "\nWeekly output:"
)


print(
    WEEKLY_DIR
)


print(
    "\nSample:"
)


print(
    "10 treatment units"
)


print(
    "5 counterfactual units per treatment"
)


print(
    "50 counterfactual units"
)


print(
    "60 total spatial units"
)


print(
    "\nTemporal design:"
)


print(
    "20 BEFORE × 7 days"
)


print(
    "20 AFTER × 7 days"
)


print(
    "\nBefore:"
)


print(
    "2024-05-10 through 2024-09-26"
)


print(
    "\nAfter:"
)


print(
    "2024-09-27 through 2025-02-13"
)


print(
    "\nExpected weekly panel rows:"
)


print(
    "Per sensor:",
    EXPECTED_TOTAL_SITES
    *
    EXPECTED_WEEKS_TOTAL
)


print(
    "Both sensors:",
    EXPECTED_TOTAL_SITES
    *
    EXPECTED_WEEKS_TOTAL
    *
    2
)


print(
    "\nWeekly image quality:"
)


print(
    QUALITY_CSV_FILE
)


print(
    "\nWeekly Excel:"
)


print(
    QUALITY_EXCEL_FILE
)


print(
    "\nSummary:"
)


print(
    SUMMARY_CSV_FILE
)


print(
    "\nPeriod definitions:"
)


print(
    PERIOD_DEFINITION_FILE
)


print(
    "\nSample validation:"
)


print(
    SAMPLE_VALIDATION_FILE
)


print(
    "\nExisting valid weekly TIFFs are reused:"
)


print(
    SKIP_EXISTING_WEEKLY
)


print(
    "\nNo Earth Engine download occurs in this notebook."
)


print(
    "\nNotebook completed successfully."
)

Packages loaded successfully.

Notebook 10 daily inventory:
/Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/daily_datasets/daily_satellite_inventory.csv

Notebook 10 selected sample:
/Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/daily_datasets/selected_site_sample.csv

Study period:
2024-05-10 through 2025-02-13

Before:
2024-05-10 through 2024-09-26

After:
2024-09-27 through 2025-02-13

Weekly output directory:
/Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/weekly_datasets

SELECTED SAMPLE

Treatment sites: 10
Counterfactual sites: 50
Total sites: 60

Controls per treatment:
matched_treatment_site_id  counterfactual_count  expected_counterfactual_count  sample_complete
           treatment_0001                     5                              5             True
           treatment_0002                     5                              5             True
           treatment_0003                